<div align="center">

# 🚕 Yandex Go Analytics

## 1 · Setup

*Build the catalog skeleton, create the landing Volume and generate the mock source files.*

![layer](https://img.shields.io/badge/layer-raw%20files-6C757D)
![tooling](https://img.shields.io/badge/tooling-Asset%20Bundles-4B8BBE)
![data](https://img.shields.io/badge/data-Faker-2E7D32)

</div>

| ◀ Previous | 🏠 Home | Next ▶ |
|:--|:--:|--:|
| [0 · Project Overview](./0.%20Project%20Overview.ipynb) | [Project Overview](./0.%20Project%20Overview.ipynb) | [2 · Batch Ingestion](./2.%20Batch%20Ingestion.ipynb) |

**Progress** &nbsp; ▰▱▱▱▱▱▱▱  **1 / 8**

---

## 🎯 Task requirements

- [x] Create four schemas in the catalog: `raw_files`, `bronze`, `silver`, `gold`
- [x] Create a **Unity Catalog Volume** inside `raw_files`
- [x] Provide mock source files — **Parquet** (taxi trips) and **Avro** (users) — inside the Volume

## 🧩 How it is solved

| Requirement | Implementation |
|:--|:--|
| Schemas + Volume | Declared as code in `resources/schemas.yml` and `resources/volumes.yml`, deployed with **Databricks Asset Bundles** |
| Source files | Faker generators write files straight into the Volume: `parquet/` (trips), `avro/` (users), `json/` (drivers) |

```text
/Volumes/<catalog>/raw_files/landing/
├── parquet/   ← 🚕 taxi trips   (Parquet)
├── avro/      ← 👤 users        (Avro)
└── json/      ← 🧑‍✈️ drivers      (JSON, used by the stream in notebook 3)
```

## 1️⃣ Deploy the infrastructure

Run once from the project root. The bundle creates the schemas, the Volume, the DLT pipeline and the scheduled job.

```bash
make validate      # databricks bundle validate -t dev
make deploy-dev    # databricks bundle deploy   -t dev
```

> ⚠️ **Prerequisite — the `innowise` group.** `resources/schemas.yml` grants the `innowise` group access to the
> Gold schema and `resources/pipelines/*.yml` gives it `CAN_VIEW` on the pipeline, so the group must exist at
> deploy time. If your workspace does not have it yet, create it first (notebook 8, section 1 does this
> idempotently) or temporarily comment out the `grants` / `permissions` blocks.

Then continue below to **verify** what was deployed.

## 2️⃣ Connect to Spark

One bootstrap cell: it makes `src` importable, loads `.env`, and opens a Spark session
(serverless Databricks Connect locally, the active session inside Databricks).

In [13]:
from pyspark.sql import functions as F

from src.core.config import get_catalog, get_environment
from src.core.infra import DatabricksConnectionService

conn = DatabricksConnectionService(app_name="yandex-go-setup")
conn.connect()
spark = conn.spark

CATALOG = get_catalog()
print(f"✅ Connected | Spark {spark.version} | catalog = {CATALOG} | env = {get_environment()}")

2026-10-09 15:55:45.050 | INFO     | src.core.infra.connection.databricks_connector:connect:56 - SparkSession ready | app=yandex-go-setup | version=4.2.0 | runtime=serverless-connect


✅ Connected | Spark 4.2.0 | catalog = yandex_go_staging | env = staging


## 3️⃣ Verify the catalog structure

### 🗂 Schemas

In [14]:
EXPECTED_SCHEMAS = {"raw_files", "bronze", "silver", "gold"}

schemas_df = spark.sql(f"SHOW SCHEMAS IN {CATALOG}")
display(schemas_df.toPandas())

found = {row[0] for row in schemas_df.collect()}
missing = EXPECTED_SCHEMAS - found
assert not missing, f"Missing schemas: {sorted(missing)} - run `make deploy-dev` first"
print(f"✅ All four schemas exist in `{CATALOG}`: {sorted(EXPECTED_SCHEMAS)}")

,databaseName
0,bronze
1,default
2,gold
3,information_schema
4,raw_files
5,silver
6,trips_dlt


✅ All four schemas exist in `yandex_go_staging`: ['bronze', 'gold', 'raw_files', 'silver']


### 📦 Unity Catalog Volume

In [15]:
display(spark.sql(f"SHOW VOLUMES IN {CATALOG}.raw_files").toPandas())

display(spark.sql(f"DESCRIBE VOLUME {CATALOG}.raw_files.landing").toPandas())
print(f"✅ Volume `{CATALOG}.raw_files.landing` is ready")

,database,volume_name
0,raw_files,landing


,name,catalog,database,owner,storage_location,volume_type,comment,securable_type,securable_kind
0,landing,yandex_go_staging,raw_files,stanislav.panfilenko@gmail.com,,MANAGED,"Landing volume for Parquet, Avro and Stream JSONs",VOLUME,VOLUME_DB_STORAGE


✅ Volume `yandex_go_staging.raw_files.landing` is ready


## 4️⃣ Generate the mock source files

Three generators simulate files arriving from upstream systems. They share **deterministic ID pools**
(fixed Faker seeds), so trips always reference users and drivers that really exist.

| Entity | Generator | Format | Rows | Deliberate anomalies |
|:--|:--|:--:|--:|:--|
| 🧑‍✈️ Drivers | `DriverGenerator` | JSON | 50 | zero experience, rating `0.5` / `5.8` (~8 %) |
| 👤 Users | `UserGenerator` | Avro | 500 | null name or e-mail (~6 %) |
| 🚕 Trips | `TripGenerator` | Parquet | 2 000 | zero passengers / distance / fare (~5 %) |

> 🎯 The anomalies are intentional — they give every Silver quality rule something to catch.

### 🧑‍✈️ Drivers

In [16]:
from src.core.utils.data_generation.driver import DriverGenerator

print("Generating mock drivers data...")

driver_gen = DriverGenerator(subfolder="json", volume_name="landing", pool_size=100)
driver_gen.generate(spark, row_count=50, with_phone=False)

print(f"✅ Drivers written to: {driver_gen.output_path}")

Generating mock drivers data...
✅ Drivers written to: /Volumes/yandex_go_staging/raw_files/landing/json


### 👤 Users (Avro)

In [17]:
from src.core.utils.data_generation.user import UserGenerator

print("Generating mock user profiles...")

user_gen = UserGenerator(subfolder="avro", volume_name="landing", pool_size=500)
user_gen.generate(spark, row_count=500)

print(f"✅ Users written to: {user_gen.output_path}")

Generating mock user profiles...
✅ Users written to: /Volumes/yandex_go_staging/raw_files/landing/avro


### 🚕 Taxi trips (Parquet)

In [18]:
from src.core.utils.data_generation.trip import TripGenerator

print("Generating mock taxi trips...")

trip_gen = TripGenerator(
    subfolder="parquet", volume_name="landing", user_pool_size=500, driver_pool_size=100
)
trip_gen.generate(spark, row_count=2000)

print(f"✅ Trips written to: {trip_gen.output_path}")

Generating mock taxi trips...
✅ Trips written to: /Volumes/yandex_go_staging/raw_files/landing/parquet


---

## ✅ Checkpoint

- [x] Four schemas exist: `raw_files`, `bronze`, `silver`, `gold`
- [x] Volume `raw_files.landing` is created
- [x] Mock Parquet, Avro and JSON files are generated inside the Volume

> **Next up ▶ [2 · Batch Ingestion](./2.%20Batch%20Ingestion.ipynb)** — ingest the Parquet and Avro files into Bronze Delta tables.

<div align="center"><sub>Yandex Go Analytics · notebook 1 of 8 · <a href="https://github.com/stanislavpanfilenko/YandexGoAnalytics">GitHub</a>